In [2]:
import config as fg
import pandas as pd
import ollama

In [2]:
df = pd.read_csv(fg.TERMES_PATH)

In [53]:
df

,term
0,édifice religieux
1,église
2,altération
3,pierre
4,peinture
5,cathédrale
6,peinture murale
7,stratigraphie
8,restauration
9,pigment


In [29]:
df["term"].iloc[0]

'édifice religieux'

In [6]:
prompt_1 = "Donne les concepts associés au terme : 'édifice religieux'."

In [7]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_1)

In [8]:
print(response["response"])

 Les édifices religieux sont des bâtiments construits pour la pratique de différentes religions. Voici quelques concepts associés à ce terme :

1. Église : Bâtiment de culte chrétien, utilisé pour les services religieux et les événements religieux tels que le mariage ou la baptême.

2. Temple : Un édifice dédié aux cultes du judaïsme, de l'hindouisme, du bouddhisme et de l'islam, entre autres.

3. Mosquée : Bâtiment où se pratiquent les prières musulmanes dans la religion islamique.

4. Synagogue : Édifice de culte juif utilisé pour la prière et les services religieux, ainsi que pour des activités communautaires.

5. Cathédrale : Une grande église qui est le siège d'un évêque dans une diocèse catholique romain ou d'une cathédrale anglicane dans l'Église d'Angleterre.

6. Basilique : Bâtiment de style chrétien, souvent grand et majestueux, utilisé pour les offices religieux.

7. Minaret : Tour élevée à l'extérieur d'une mosquée, servant à la fois de symbole architectural et de tour pour

Analyse du résultat

✅ Points positifs

- Le modèle a bien compris le terme "édifice religieux"
- Il génère des concepts narrower (plus spécifiques) pertinents
- Les définitions sont correctes et compréhensibles
- Bonne couverture multi-religieuse (islam, judaïsme, christianisme, bouddhisme)


❌ Problèmes identifiés
1. Format non structuré
Le résultat est du texte libre, pas structuré — difficile à traiter automatiquement dans le pipeline TAC.
2. Mélange de concepts

- Minaret n'est pas un édifice religieux, c'est une partie d'une mosquée
- Mausolée est discutable comme édifice religieux
- Temple est trop générique (mélange plusieurs religions)

3. Manque des relations SKOS importantes

❌ Pas de broader (concept plus large)
❌ Pas de related (concepts associés)
❌ Pas de altLabel (synonymes)

In [12]:
prompt_2 = """ 
Pour le terme "édifice religieux", donne :

1. Les définitions (uniquement les définitions qui font référence à des bâtiments complets, pas des parties d'un bâtiment)
2. Les termes alternatifs (synonymes)
3. Le concept plus large
4. Les concepts plus spécifiques (uniquement des bâtiments complets, pas des parties d'un bâtiment)
5. Les concepts associés

Réponds sous forme de liste structurée, sans explication supplémentaire.
"""

In [13]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_2)

In [14]:
print(response["response"])

 1. Définitions:
   - Building used for religious worship or ceremonies (généralement)
   - A place of worship, especially a Christian church (plus spécifique pour le christianisme)
   - Mosque (Islam)
   - Temple (Hinduism, Buddhism)
   - Synagogue (Judaism)

2. Termes alternatifs:
   - Place of worship
   - House of God
   - Religious structure

3. Concept plus large:
   - Religious architecture
   - Sacred space

4. Concepts plus spécifiques:
   - Basilica (un type de cathédrale)
   - Minaret (partie du mosquée où le muazzin appelle au prière)
   - Mandir (temple hindouiste en Inde)

5. Concepts associés:
   - Iconographie religieuse
   - Art sacré
   - Rituals religieux
   - Symboles religieux
   - Reliques


Analyse du résultat :

✅ Points positifs

- Mandir est un bon ajout pertinent
- Les concepts associés sont plus propres
- Structure respectée

❌ Problèmes majeurs
1. Langue incohérente — gros problème
Le modèle mélange français et anglais dans la même réponse :

- Building used for religious worship → anglais
- Place of worship → anglais
- Iconographie religieuse → français

On n'a pas précisé la langue dans le prompt !
2. Définitions mal comprises
Le modèle liste des types d'édifices comme définitions :

- Mosque (Islam) → c'est un narrower, pas une définition
- Temple (Hinduism) → pareil
- Ce n'est pas du tout ce qu'on attendait

3. Minaret encore présent !

❌ Minaret dans les concepts spécifiques
On l'a déjà signalé deux fois, le prompt ne l'empêche pas assez clairement

4. Termes alternatifs en anglais

Place of worship, House of God → pas en français




In [15]:
prompt_3 = """
Tu es un expert en thésaurus pour le patrimoine culturel.
Réponds uniquement en français.

Pour le terme "édifice religieux", donne :

1. Définition : une seule phrase qui décrit ce qu'est un édifice religieux
   en général, sans citer de types spécifiques

2. Termes alternatifs : uniquement des synonymes directs en français
   (pas des sous-types, pas de noms en anglais)

3. Concept plus large : le domaine auquel appartient ce terme (1 seul)

4. Concepts plus spécifiques : uniquement des types de bâtiments religieux
   complets en français, pas des parties d'un bâtiment comme un minaret

5. Concepts associés : maximum 4, en français, courts et simples

Réponds sous forme de liste structurée, sans explication supplémentaire.
"""

In [16]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_3)

In [17]:
print(response["response"])

 1. Définition : Bâtiment dédié à la pratique d'une religion ou au culte d'un dieu ou d'un saint.
2. Termes alternatifs : Église, temple, mosquée, synagogue, basilique, cathédrale, établissement de culte.
3. Concept plus large : Architecture religieuse.
4. Concepts plus spécifiques : Cathédrale gothique, mosquée ottomane, temple hindou, synagogue juive, basilique byzantine.
5. Concepts associés : Autel, statues de divinités, vitraux, chœur, clocher, orgue.


Analyse du résultat :

✅ Améliorations notables

✅ Tout en français cette fois

✅ Définition correcte et concise

✅ Concept plus large pertinent : Architecture religieuse

✅ Structure respectée


❌ Problèmes qui persistent
1. Termes alternatifs encore mal compris

- Église, Mosquée, Cathédrale, Basilique → ce sont des narrower, pas des synonymes
- Les vrais synonymes : Bâtiment cultuel, Monument sacré, Lieu de culte

2. Concepts spécifiques trop composés

- Cathédrale gothique → mélange type + style architectural
- Mosquée ottomane → mélange type + période historique
- Synagogue juive → redondant, une synagogue est forcément juive

3. Concepts associés = parties d'un bâtiment

❌ Autel, Vitraux, Chœur, Clocher, Orgue → ce sont des éléments intérieurs/extérieurs d'un édifice, pas des concepts associés
Les vrais concepts associés : Art sacré, Patrimoine culturel, Archéologie religieuse

In [26]:
prompt_4 = """
Tu es un expert en thésaurus pour le patrimoine culturel.
Réponds uniquement en français.

Pour le terme "édifice religieux", donne :

1. Définition : Une définition générales décrivant le terme, sans citer de types spécifiques.

2. Termes alternatifs : uniquement des synonymes directs qui peuvent
   remplacer le terme dans une phrase
   ❌ PAS des sous-types comme : Église, Mosquée, Cathédrale, ...
   ✅ OUI des synonymes comme : Bâtiment cultuel, Monument sacré, ...

3. Concept plus large : 1 seul concept plus général auquel appartient le terme

4. Concepts plus spécifiques : uniquement des types génériques de bâtiments
   complets, pas des combinaisons type + style ou type + religion
   ❌ PAS : Cathédrale gothique, Mosquée ottomane, Synagogue juive, ...
   ✅ OUI : Cathédrale, Mosquée, Synagogue, Abbaye, ...

5. Concepts associés : domaines ou disciplines liés au terme,
   pas des parties physiques du bâtiment
   ❌ PAS : Autel, Vitraux, Clocher, Orgue, ...
   ✅ OUI : Art sacré, Patrimoine culturel, Architecture, ...

Réponds sous forme de liste structurée, sans explication supplémentaire.
"""

In [27]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_4)

In [28]:
print(response["response"])

 1. Définition : Un bâtiment ou édifice destiné à la pratique de la religion et aux cérémonies religieuses.
2. Termes alternatifs : Bâtiment cultuel, Monument sacré, Sanctuaire, Établissement religieux, Lieu de culte.
3. Concept plus large : Architecture, Bâtiments cultuels et d'occupation spéciale.
4. Concepts plus spécifiques : Cathédrale, Mosquée, Synagogue, Abbaye, Temple, Église, Chapelle, Basilique, Monastère.
5. Concepts associés : Art sacré, Patrimoine culturel, Religion, Histoire de l'art, Architecture religieuse, Symbolisme, Iconographie religieuse.


Analyse du résultat :

✅ Grandes améliorations

✅ Définition correcte et générale, sans citer de types

✅ Termes alternatifs enfin corrects : Bâtiment cultuel, Monument sacré, Lieu de culte

✅ Concepts spécifiques propres et génériques

✅ Concepts associés pertinents et sans parties physiques

✅ Tout en français

✅ Zéro hallucination



⚠️ Petits points à discuter
1. Concept plus large — trop précis

- Bâtiments cultuels et d'occupation spéciale → trop technique et bizarre
- On attendait juste : Architecture ou Patrimoine architectural

2. Concepts associés — un peu trop nombreux

- 6 concepts associés c'est beaucoup
- Architecture religieuse devrait être dans le broader pas ici

In [33]:
def generate_prompt(terme: str) -> str:
    return f"""
Tu es un expert en thésaurus pour le patrimoine culturel.
Réponds uniquement en français.

Pour le terme "{terme}", donne :

1. Définition : une phrase générale décrivant le terme, sans citer de types spécifiques

2. Termes alternatifs : uniquement des synonymes directs qui peuvent
   remplacer le terme dans une phrase
   ❌ PAS des sous-types comme : Église, Mosquée, Cathédrale, ...
   ✅ OUI des synonymes comme : Bâtiment cultuel, Monument sacré, ...

3. Concept plus large : 1 seul concept très général
   ❌ PAS un concept trop précis
   ✅ OUI un concept simple comme : Architecture, Patrimoine architectural

4. Concepts plus spécifiques : uniquement des types génériques complets,
   pas des combinaisons type + style ou type + religion
   ❌ PAS : Cathédrale gothique, Mosquée ottomane, ...
   ✅ OUI : Cathédrale, Mosquée, Synagogue, Abbaye, ...

5. Concepts associés : maximum 4 domaines ou disciplines liés au terme,
   pas des parties physiques, pas le concept plus large
   ❌ PAS : Autel, Vitraux, Architecture religieuse, ...
   ✅ OUI : Art sacré, Patrimoine culturel, Religion, ...

Réponds sous forme de liste structurée, sans explication supplémentaire.
"""


In [42]:
prompt_5 = generate_prompt(df["term"].iloc[0])

In [35]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_5)

In [36]:
print(response["response"])

 1. Définition : Bâtiment consacré à des cérémonies religieuses ou spirituelles.
2. Termes alternatifs : Bâtiment cultuel, Monument sacré, Sanctuaire, lieu de culte, église consacrée.
3. Concept plus large : Architecture, Patrimoine architectural.
4. Concepts plus spécifiques : Cathédrale, Mosquée, Synagogue, Abbaye, Église orthodoxe, Temple hindou, Maison de prière, Monastère.
5. Concepts associés : Art sacré, Patrimoine culturel, Religion, Histoire religieuse, Théologie, Archéologie religieuse, Iconographie religieuse, Symbolisme religieux.


Analyse du résultat :

✅ Points positifs

✅ Définition correcte et générale, sans citer de types

✅ Termes alternatifs globalement bons : Bâtiment cultuel, Monument sacré, Lieu de culte

✅ Broader correct cette fois : Architecture, Patrimoine architectural

✅ Narrower propres et génériques

✅ Zéro hallucination


❌ Petits problèmes

1. Termes alternatifs — un peu douteux

Église consacrée → trop spécifique, c'est un narrower

2. Concepts spécifiques — encore des combinaisons

Église orthodoxe → type + style ❌

Temple hindou → type + religion ❌

In [43]:
response = ollama.generate(model=fg.MODEL_LLAMA, prompt=prompt_5)

In [44]:
print(response["response"])

Voici les réponses :

1. Définition
   - Un bâtiment construit pour servir à des fins religieuses.

2. Termes alternatifs
   - Bâtiment cultuel
   - Monument sacré
   - Lieu de culte
   - Édifice sanctuaire

3. Concept plus large
   - Patrimoine culturel

4. Concepts plus spécifiques
   - Cathédrale
   - Mosquée
   - Synagogue
   - Abbaye
   - Temple

5. Concepts associés
   - Art sacré
   - Patrimoine culturel
   - Religion
   - Histoire


Analyse — Llama 2 vs Mistral

✅ Points positifs

✅ Définition courte et correcte

✅ Termes alternatifs bons : Bâtiment cultuel, Monument sacré, Lieu de culte

✅ Narrower propres et génériques

✅ Concepts associés courts et pertinents

✅ Résultat stable et cohérent

✅ Respect du nombre demandé


❌ Problèmes

1. Broader incorrect

Patrimoine culturel → c'est un concept associé, pas un broader

✅ OUI : Architecture ou Patrimoine architectural

2. Concepts associés — doublon avec broader

Patrimoine culturel apparaît dans broader ET concepts associés en même temps

❌ Contradiction dans la réponse

In [48]:
df["term"].iloc[1]

'église'

In [45]:
prompt_5 = generate_prompt(df["term"].iloc[1])

In [46]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_5)

In [47]:
print(response["response"])

 1. Définition : Bâtiment consacré à la célébration des offices religieux et servant d'édifice central au culte.

2. Termes alternatifs : Bâtiment cultuel, Monument sacré, Maison de Dieu, Lieu de culte, Sanctuaire, Édifice religieux.

3. Concept plus large : Architecture, Patrimoine architectural, Art sacré, Culture matérielle.

4. Concepts plus spécifiques : Cathédrale, Mosquée, Synagogue, Abbaye, Temple, Basilique.

5. Concepts associés : Histoire de l'art, Religion, Patrimoine culturel, Art religieux, Iconographie, Architecture religieuse, Théologie, Spiritualité.


Analyse :

✅ Points positifs

✅ Définition correcte et précise pour "église"

✅ Termes alternatifs globalement bons

✅ Tout en français

✅ Zéro hallucination


❌ Problèmes

1. Termes alternatifs — confusion

- Édifice religieux → c'est le broader d'église, pas un synonyme !
- Sanctuaire → discutable comme synonyme direct

2. Broader — trop nombreux

- On attendait 1 seul concept plus large
- Le modèle a donné 4 : Architecture, Patrimoine architectural, Art sacré, Culture matérielle
✅ OUI : juste Édifice religieux ou Patrimoine architectural

3. Narrower — mélange de religions

- Mosquée, Synagogue, Temple → ce sont des narrower d'édifice religieux, pas d'église
- Une église est spécifiquement chrétienne, donc les narrower corrects seraient : Cathédrale, Basilique, Chapelle, Abbaye

4. Concepts associés — trop nombreux et redondants

- 8 concepts au lieu de 4 maximum
- Architecture religieuse → déjà dans le broader
- Art religieux et Art sacré → doublon

In [56]:
df["term"].iloc[9]

'pigment'

In [50]:
prompt_5 = generate_prompt(df["term"].iloc[9])

In [51]:
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_5)

In [52]:
print(response["response"])

 1. Définition : Une substance minérale, organique ou synthétique qui donne sa couleur à un matériau lorsqu'elle est mélangée avec lui.

2. Termes alternatifs : Couleurant, colorant, teinture, pigment naturel, pigment artificiel, ocre, terre de Sienna, ultramarine, cobalt, jaune d'œuf, carmine, etc.

3. Concept plus large : Architecture, Patrimoine architectural, Design, Art, Science des matériaux, etc.

4. Concepts plus spécifiques : Cathédrale, Mosquée, Synagogue, Abbaye, Palais, Maison, Temple, Chapelle, Église, etc.

5. Concepts associés : Art sacré, Patrimoine culturel, Histoire de l'art, Coloration, Peinture, Artisanat, Textile, Industrie chimique, Photographie, etc.


Analyse:

❌ Résultat catastrophique !

Problèmes majeurs

1. Définition — seul point correct

- ✅ La définition est bonne et précise

2. Termes alternatifs — mélange de tout

- Pigment naturel, Pigment artificiel → ce sont des narrower, pas des synonymes
- Ocre, Terre de Sienna, Ultramarine, Cobalt → ce sont des types de pigments, pas des synonymes
- Jaune d'œuf → c'est un liant, pas un pigment ❌
- Seuls Colorant, Teinture sont acceptables comme synonymes

3. Broader — complètement faux

- Architecture, Patrimoine architectural, Design → aucun rapport avec pigment ❌
- ✅ OUI : Matière colorante ou Chimie des couleurs

4. Narrower — COMPLÈTEMENT FAUX

- Cathédrale, Mosquée, Synagogue, Abbaye → ce sont des narrower d'édifice religieux !!
- Le modèle a copié-collé les narrower du terme précédent ❌
- ✅ OUI : Pigment minéral, Pigment organique, Pigment synthétique

5. Concepts associés — hors sujet

- Art sacré, Patrimoine culturel → liés à édifice religieux, pas à pigment ❌
- Seuls Peinture, Coloration, Textile sont pertinents

In [55]:
df["term"].iloc[20]

'sculptrice'

In [57]:
prompt_5 = generate_prompt(df["term"].iloc[20])
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt_5)
print(response["response"])

 1. Définition : Une femme spécialisée dans la sculpture, créant des œuvres d'art en trois dimensions.

2. Termes alternatifs :
   - Statuaire
   - Artiste de cire perdue
   - Faussaire
   - Sculptrice sur bois

3. Concept plus large : Architecture

4. Concepts plus spécifiques :
   - Cathédrale
   - Mosquée
   - Synagogue
   - Abbaye

5. Concepts associés :
   - Art sacré
   - Patrimoine culturel
   - Historique de l'art
   - Femmes en art


Analyse :

❌ Résultat très problématique !

Problèmes majeurs
1. Définition — correcte mais incomplète

- ✅ La définition est acceptable
- ⚠️ Mais sculptrice peut aussi désigner un homme selon les contextes modernes

2. Termes alternatifs — hallucinations graves

- Faussaire → aucun rapport avec sculptrice ❌ hallucination
- Artiste de cire perdue → c'est une technique, pas un synonyme ❌
- Sculptrice sur bois → c'est un narrower, pas un synonyme ❌
- ✅ OUI : Sculpteur, Plasticienne, Artiste plasticien

3. Broader — complètement faux

- Architecture → aucun rapport avec sculptrice ❌
- ✅ OUI : Artiste, Plasticien, Art visuel

4. Narrower — copié d'édifice religieux !

- Cathédrale, Mosquée, Synagogue, Abbaye → exactement les mêmes narrower qu'édifice religieux ❌
- Contamination du contexte encore plus grave que pour "pigment"
- ✅ OUI : Sculptrice sur bois, Sculptrice sur pierre, Sculptrice contemporaine

5. Concepts associés — mitigés

- ✅ Femmes en art → pertinent et intéressant
- ✅ Histoire de l'art → pertinent
- ❌ Art sacré, Patrimoine culturel → copiés d'édifice religieux